In [ ]:
import time
import random
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torchvision.transforms import v2
from torch.utils.tensorboard import SummaryWriter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 4

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [ ]:
def imshow(img):
    img = img / 2 + 0.5
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


dataiter = iter(trainloader)
images, labels = next(dataiter)

imshow(torchvision.utils.make_grid(images))
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(batch_size)))

In [ ]:
class TutorialNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.conv3 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv4 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 4096)
        self.fc2 = nn.Linear(4096, 1000)
        self.fc3 = nn.Linear(1000, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(F.relu(self.conv2(x)))
        x = F.relu(self.conv3(x))
        x = self.pool(F.relu(self.conv4(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


class DeepNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
def train(net, device, writer, epochs=2, lr=0.001, loader=None, log_every=2000):
    loader = loader or trainloader
    net.to(device)
    net.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    start = time.perf_counter()
    for epoch in range(epochs):
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(loader, 0):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = net(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if i % log_every == log_every - 1:
                print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / log_every:.3f}')
                writer.add_scalar('training loss', running_loss / log_every,
                                  (epoch * len(loader) + i) * loader.batch_size)
                running_loss = 0.0
    if device.type == 'cuda':
        torch.cuda.synchronize()
    duration = time.perf_counter() - start
    writer.flush()
    print(f'Finished Training on {device} in {duration:.1f} s')
    return duration


def evaluate(net, device, k=3):
    net.to(device)
    net.eval()
    top1 = topk = total = 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = net(images)
            top = outputs.topk(k, dim=1).indices
            top1 += (top[:, 0] == labels).sum().item()
            topk += (top == labels.unsqueeze(1)).any(dim=1).sum().item()
            total += labels.size(0)
    print(f'Top-1 accuracy: {100 * top1 / total:.2f} %')
    print(f'Top-{k} accuracy: {100 * topk / total:.2f} %')
    return topk / total

In [ ]:
tutorial_net = TutorialNet()
tutorial_writer = SummaryWriter('runs/cifar10/tutorial_net')
tutorial_writer.add_graph(tutorial_net, images)
train(tutorial_net, device, tutorial_writer)
evaluate(tutorial_net, device)
tutorial_writer.close()

In [ ]:
torch.manual_seed(0)
net_cpu = Net()
cpu_writer = SummaryWriter('runs/cifar10/net_cpu')
cpu_writer.add_graph(net_cpu, images)
cpu_time = train(net_cpu, torch.device('cpu'), cpu_writer)
cpu_writer.close()

In [ ]:
torch.manual_seed(0)
net = Net()
gpu_writer = SummaryWriter('runs/cifar10/net_gpu')
gpu_writer.add_graph(net, images)
gpu_time = train(net, device, gpu_writer)

print(f'CPU: {cpu_time:.1f} s')
print(f'GPU: {gpu_time:.1f} s')
print(f'Speedup: {cpu_time / gpu_time:.2f}x')

In [ ]:
evaluate(net, device)

In [ ]:
def top3_errors(net, device, n=5):
    net.to(device)
    net.eval()
    errors = []
    with torch.no_grad():
        for images, labels in testloader:
            outputs = net(images.to(device)).cpu()
            top = outputs.topk(3, dim=1).indices
            for img, label, pred in zip(images, labels, top):
                if label not in pred:
                    errors.append((img, label.item(), pred.tolist()))
    return random.sample(errors, n)


samples = top3_errors(net, device)

fig = plt.figure(figsize=(15, 4))
for idx, (img, label, pred) in enumerate(samples):
    ax = fig.add_subplot(1, 5, idx + 1, xticks=[], yticks=[])
    ax.imshow(np.transpose((img / 2 + 0.5).numpy(), (1, 2, 0)))
    ax.set_title(f'true: {classes[label]}\npred: {", ".join(classes[p] for p in pred)}')
gpu_writer.add_figure('top-3 errors', fig)
gpu_writer.add_images('top-3 error images', torch.stack([s[0] for s in samples]) / 2 + 0.5)
gpu_writer.add_text('top-3 error labels', '  \n'.join(
    f'{i}: true={classes[l]}, pred={[classes[p] for p in pr]}' for i, (_, l, pr) in enumerate(samples)))
gpu_writer.close()

In [ ]:
train_transform = v2.Compose([
    v2.ToImage(),
    v2.RandomCrop(32, padding=4),
    v2.RandomHorizontalFlip(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
deep_trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                             download=True, transform=train_transform)
deep_trainloader = torch.utils.data.DataLoader(deep_trainset, batch_size=128,
                                               shuffle=True, num_workers=4)

deep_net = DeepNet()
deep_writer = SummaryWriter('runs/cifar10/deep_net')
deep_writer.add_graph(deep_net, images)
train(deep_net, device, deep_writer, epochs=15, lr=0.01, loader=deep_trainloader, log_every=100)
evaluate(deep_net, device)
deep_writer.close()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/cifar10